In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPRegressor
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR
from sklearn import ensemble
from sklearn.metrics import mean_squared_error
import time

In [ ]:
# Load Data
df = pd.DataFrame(pd.read_csv(r"C:\Users\20214107\OneDrive - TU Eindhoven\Desktop\Data analytics\Assignment 01\Data\Integrated_data_RevisedClean_Families_HighCorrel.csv", header=0, index_col=0, parse_dates=True, squeeze=True))

In [ ]:
# Split features and target
X = df.drop('nextday_DAIRY_sales', axis=1)
y = df['nextday_DAIRY_sales'].values

# Dropping unnecessary features
X = X.drop(['day_of_month', ], axis = 1)

In [ ]:
# OnehotEncoding and KNN imputation

# OneHotEncoding
categorical_features = ['store_nbr', 'month', 'year', 'cluster']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object', 'bool']).columns

categorical_features.extend(cat_features.tolist())

for cat in categorical_features:
    one_hot = pd.get_dummies(X[cat], prefix=cat)
    X[one_hot.columns] = one_hot
X = X.drop(categorical_features, axis=1)

# KNN Imputation
numeric_features = numeric_features.drop(['store_nbr', 'month', 'year', 'cluster'])
numeric_transformer = Pipeline(steps=[("imputer", KNNImputer(n_neighbors=5, weights="uniform"))])
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features)], remainder='passthrough')

Final_X = pd.DataFrame(preprocessor.fit_transform(X))
Final_X.columns = X.columns

C:\Users\20214107\Anaconda3\lib\site-packages\pandas\core\frame.py:3641: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  self[k1] = value[k2]


In [ ]:
# Normalized dataset
numeric_transformer = Pipeline(steps=[("scaler", MinMaxScaler())])
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features)], remainder='passthrough')
Data_norm = pd.DataFrame(preprocessor.fit_transform(Final_X))
Data_norm.columns = Final_X.columns

In [ ]:
# Normalized labels
y = y.reshape(len(y),1)
sc_y = MinMaxScaler()
y = sc_y.fit_transform(y)

In [ ]:
# Data split
X_train, X_test, y_train, y_test = train_test_split(Data_norm, y, test_size = 0.2, shuffle=False, random_state=42)

Support vector regressor

In [ ]:
# param = {'kernel':('linear', 'poly', 'rbf', 'sigmoid'), 'C':[1, 5, 10], 'degree':[3, 8], 'coef0':[0.01, 0.1, 0.5], 'gamma':('auto', 'scale')}
# kernel: Specifies the kernel type to be used in the algorithm.
# C: Regularization parameter. Tells the SVM optimization how much you want to avoid misclassifying each training example.
# degree: Degree of the polynomial kernel function.
# coef0: Independent term in kernel function.
# gamma: Kernel coefficient for ‘rbf’, ‘poly’ and ‘sigmoid’.
# https://medium.com/machine-learning-101/chapter-2-svm-support-vector-machine-theory-f0812effc72#:~:text=The%20Regularization%20parameter%20(often%20termed,avoid%20misclassifying%20each%20training%20example.

param = {'kernel':['linear'], 'C':[1], 'degree':[3], 'coef0':[0.01], 'gamma':['auto']}

svr_regressor = SVR()

# Grid search
start_hp_svr = time.time()
svr_grids = GridSearchCV(estimator=svr_regressor, param_grid = param, n_jobs=-1).fit(X_train, y_train)

print(" Results from Grid Search " )
print("\n hyperparameter tuning time:\n", time.time()-start_hp_svr)
print("\n The best estimator across ALL searched params:\n", svr_grids.best_estimator_)
print("\n The best score across ALL searched params:\n", svr_grids.best_score_)
print("\n The best parameters across ALL searched params:\n", svr_grids.best_params_)

predicted_mlp = svr_grids.predict(X_test)
print(f"\n The root square error for test set is:\n {svr_grids.score(X_test, y_test)}")
print(f"\n The mean square error for test set is:\n {mean_squared_error(y_test, predicted_mlp)}")

C:\Users\20214107\Anaconda3\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


 Results from Grid Search 

 The best estimator across ALL searched params:
 SVR(C=1, coef0=0.01, gamma='auto', kernel='linear')

 The best score across ALL searched params:
 0.8212649218571022

 The best parameters across ALL searched params:
 {'C': 1, 'coef0': 0.01, 'degree': 3, 'gamma': 'auto', 'kernel': 'linear'}

 The root square error for test set is:
0.7731456036379734

 The mean square error for test set is:
0.0039693758999514465


Multi-layer perceptron

In [ ]:
param = {'solver':('lbfgs', 'sgd', 'adam'), 'max_iter':[200, 500, 800], 'learning_rate':['constant', 'adaptive'], 'alpha':[0.0001, 0.05], 'hidden_layer_sizes':[(10,), (20,)]}
# solver: The solver for weight optimization.
# max_iter: Maximum number of iterations.
# learning_rate: Learning rate schedule for weight updates.
# alpha: L2 penalty (regularization term) parameter. (https://scikit-learn.org/stable/auto_examples/neural_networks/plot_mlp_alpha.html)
# hidden_layer_sizes: The ith element represents the number of neurons in the ith hidden layer.

# param = {'solver':['lbfgs'], 'max_iter':[200], 'learning_rate':['constant'], 'alpha':[0.0001], 'hidden_layer_sizes':[(10,)]}

mlp_regressor = MLPRegressor()

# Grid search
start_hp_mlp = time.time()
mlp_grids = GridSearchCV(estimator=mlp_regressor, param_grid = param, n_jobs=-1).fit(X_train, y_train)

print(" Results from Grid Search " )
print("\n hyperparameter tuning time:\n", time.time()-start_hp_mlp)
print("\n The best estimator across ALL searched params:\n", mlp_grids.best_estimator_)
print("\n The best score across ALL searched params:\n", mlp_grids.best_score_)
print("\n The best parameters across ALL searched params:\n", mlp_grids.best_params_)

start_test_mlp = time.time()
predicted_mlp = mlp_grids.predict(X_test)
print(f"\n The root square error for test set is:\n {mlp_grids.score(X_test, y_test)}")
print(f"\n The mean square error for test set is:\n {mean_squared_error(y_test, predicted_mlp)}")
print("\n Testing time:\n", time.time()-start_test_mlp)

C:\Users\20214107\Anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:1599: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


 Results from Grid Search 

 hyperparameter tuning time:
 1000.7616081237793

 The best estimator across ALL searched params:
 MLPRegressor(alpha=0.05, hidden_layer_sizes=(20,), max_iter=500)

 The best score across ALL searched params:
 0.9302462490541867

 The best parameters across ALL searched params:
 {'alpha': 0.05, 'hidden_layer_sizes': (20,), 'learning_rate': 'constant', 'max_iter': 500, 'solver': 'adam'}

 The root square error for test set is:
 0.8763525159466404

 The mean square error for test set is:
 0.0021635170010449577

 Testing time:
 0.03928422927856445


GB Model

In [ ]:
param = {"n_estimators": [100, 200, 500], "max_depth": [3, 4, 5], "min_samples_split": [2, 5], "learning_rate": [0.01, 0.1], "loss": ['squared_error', 'absolute_error']}
# n_estimators: The number of boosting stages to perform.
# max_depth: Maximum depth of the individual regression estimators. 
# min_samples_split: The minimum number of samples required to split an internal node:
# learning_rate: Learning rate shrinks the contribution of each tree.
# loss: Loss function to be optimized.
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html#sklearn.ensemble.GradientBoostingRegressor

# param = {"n_estimators": [500], "max_depth": [4], "min_samples_split": [5], "learning_rate": [0.01], "loss": ['squared_error']}

XGB_regressor = ensemble.GradientBoostingRegressor()

# Grid search
start_hp_xgb = time.time()
XGB_grids = GridSearchCV(estimator=XGB_regressor, param_grid = param, n_jobs=-1).fit(X_train, y_train)

print(" Results from Grid Search " )
print("\n hyperparameter tuning time:\n", time.time()-start_hp_xgb)
print("\n The best estimator across ALL searched params:\n", XGB_grids.best_estimator_)
print("\n The best score across ALL searched params:\n", XGB_grids.best_score_)
print("\n The best parameters across ALL searched params:\n", XGB_grids.best_params_)

start_test_xgb = time.time()
predicted_XGB = XGB_grids.predict(X_test)
print(f"\n The root square error for test set is:\n {XGB_grids.score(X_test, y_test)}")
print(f"\n The mean square error for test set is:\n {mean_squared_error(y_test, predicted_XGB)}")
print("\n Testing time:\n", time.time()-start_test_xgb)

C:\Users\20214107\Anaconda3\lib\site-packages\sklearn\ensemble\_gb.py:494: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


 Results from Grid Search 

 hyperparameter tuning time:
 6185.034572124481

 The best estimator across ALL searched params:
 GradientBoostingRegressor(max_depth=5, n_estimators=500)

 The best score across ALL searched params:
 0.9478231255677801

 The best parameters across ALL searched params:
 {'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 500}

 The root square error for test set is:
 0.8702351007484976

 The mean square error for test set is:
 0.0022705562334641243

 Testing time:
 0.25613856315612793


Bonus Data

In [ ]:
# Load data
bonus_data = pd.DataFrame(pd.read_csv(r"C:\Users\20214107\OneDrive - TU Eindhoven\Desktop\Data analytics\Assignment 01\Data\bonusQ_Integrated.csv", header=0, index_col=0, parse_dates=True, squeeze=True))

In [ ]:
for cat in categorical_features:
    one_hot = pd.get_dummies(bonus_data[cat], prefix=cat)
    bonus_data[one_hot.columns] = one_hot
bonus_data = bonus_data.drop(categorical_features, axis=1)

# KNN Imputation
# numeric_features = numeric_features.drop(['store_nbr', 'month', 'year', 'cluster'])
numeric_transformer = Pipeline(steps=[("imputer", KNNImputer(n_neighbors=5, weights="uniform"))])
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features)], remainder='passthrough')

Final_bonus = pd.DataFrame(preprocessor.fit_transform(bonus_data))
Final_bonus.columns = bonus_data.columns

C:\Users\20214107\Anaconda3\lib\site-packages\pandas\core\frame.py:3641: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  self[k1] = value[k2]


In [ ]:
Max_Dairy = bonus_data['DAIRY_sales'].max()
Min_Dairy = bonus_data['DAIRY_sales'].min()
# Normalized dataset
numeric_transformer = Pipeline(steps=[("scaler", MinMaxScaler())])
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features)], remainder='passthrough')
Bonus_norm = pd.DataFrame(preprocessor.fit_transform(Final_bonus))
Bonus_norm.columns = Final_bonus.columns

In [ ]:
season_loc = Bonus_norm.columns.get_loc('season_Winter')
for i in range(1,4):
    Bonus_norm.insert(loc=season_loc-i, column='season_%s'%i, value=0)
    season_loc += 1

month_loc = Bonus_norm.columns.get_loc('month_8')
for i in range(1,12):
    if i == 8:
        Bonus_norm.insert(loc=month_loc-i, column='month_8%s'%i, value=0)
    else:
        Bonus_norm.insert(loc=month_loc-i, column='month_%s'%i, value=0)
    month_loc += 1

year_loc = Bonus_norm.columns.get_loc('year_2017')
for i in range(1,4):
    Bonus_norm.insert(loc=year_loc-i, column='year_%s'%i, value=0)
    year_loc += 1

In [ ]:
# select only store number 1
Bonus_norm = Bonus_norm[Bonus_norm['store_nbr_1'] == 1]
# Evaluation bonus data
start_test_mlp = time.time()
predicted_bonus = mlp_grids.predict(Bonus_norm)
interval = Max_Dairy - Min_Dairy
predicted_bonus_real = (predicted_bonus * interval) + Min_Dairy
print("\n Testing time:\n", time.time()-start_test_mlp)


 Testing time:
 0.008782386779785156


C:\Users\20214107\Anaconda3\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names unseen at fit time:
- month_88
- season_1
- season_2
- season_3
- year_1
- ...
Feature names seen at fit time, yet now missing:
- month_12
- season_Autumn
- season_Spring
- season_Summer
- year_2014
- ...

  warnings.warn(message, FutureWarning)
